In [ ]:
!pip install scorecardpy

In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)
import joblib
import pickle
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.linear_model import LogisticRegression
import scorecardpy as sc
import warnings
warnings.filterwarnings('ignore')
# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Load the dataset
app_train = pd.read_csv("/kaggle/input/competitions/home-credit-default-risk/application_train.csv")

In [ ]:
app_train['DAYS_EMPLOYED'] = app_train['DAYS_EMPLOYED'].replace(365243, np.nan)
app_train['YEARS_BIRTH'] = abs(app_train['DAYS_BIRTH']) / 365
app_train['YEARS_EMPLOYED'] = abs(app_train['DAYS_EMPLOYED']) / 365

org_mapping = {
    'Business Entity Type 3': 'Business', 'Business Entity Type 2': 'Business', 'Business Entity Type 1': 'Business',
    'Government': 'Public', 'School': 'Public', 'Medicine': 'Public', 'Kindergarten': 'Public',
    'University': 'Public', 'Police': 'Public', 'Military': 'Public', 'Security Ministries': 'Public', 'Emergency': 'Public',
    'Self-employed': 'Self-employed',
    'Trade: type 7': 'Trade_Services', 'Trade: type 2': 'Trade_Services', 'Trade: type 3': 'Trade_Services',
    'Trade: type 6': 'Trade_Services', 'Trade: type 1': 'Trade_Services', 'Trade: type 5': 'Trade_Services', 'Trade: type 4': 'Trade_Services',
    'Services': 'Trade_Services', 'Hotel': 'Trade_Services', 'Restaurant': 'Trade_Services', 'Bank': 'Trade_Services',
    'Insurance': 'Trade_Services', 'Culture': 'Trade_Services', 'Legal Services': 'Trade_Services', 'Advertising': 'Trade_Services',
    'Postal': 'Trade_Services', 'Mobile': 'Trade_Services', 'Telecom': 'Trade_Services', 'Cleaning': 'Trade_Services', 'Realtor': 'Trade_Services',
    'Industry: type 11': 'Industry_Construction', 'Industry: type 1': 'Industry_Construction', 'Industry: type 4': 'Industry_Construction',
    'Industry: type 7': 'Industry_Construction', 'Industry: type 3': 'Industry_Construction', 'Industry: type 9': 'Industry_Construction',
    'Industry: type 2': 'Industry_Construction', 'Industry: type 12': 'Industry_Construction', 'Industry: type 5': 'Industry_Construction',
    'Industry: type 10': 'Industry_Construction', 'Industry: type 13': 'Industry_Construction', 'Industry: type 8': 'Industry_Construction',
    'Industry: type 6': 'Industry_Construction', 'Construction': 'Industry_Construction', 'Housing': 'Industry_Construction',
    'Agriculture': 'Industry_Construction', 'Electricity': 'Industry_Construction',
    'Transport: type 2': 'Transport', 'Transport: type 4': 'Transport', 'Transport: type 3': 'Transport', 'Transport: type 1': 'Transport',
    'Security': 'Other', 'Other': 'Other', 'XNA': 'Other', 'Religion': 'Other'
}
app_train['ORGANIZATION_TYPE'] = app_train['ORGANIZATION_TYPE'].map(org_mapping).fillna('Other')

numeric_cols = app_train.select_dtypes(include=['number']).columns
train_median = app_train[numeric_cols].median()
app_train[numeric_cols] = app_train[numeric_cols].fillna(train_median)

char_cols_train = app_train.select_dtypes(include=['object']).columns
app_train[char_cols_train] = app_train[char_cols_train].fillna('Unknown')

In [ ]:
features_to_bin = [col for col in app_train.columns if col not in ['TARGET']]
bins = sc.woebin(app_train, y='TARGET', x=features_to_bin, bin_num_limit=5)

iv_summary = sc.iv(app_train, y='TARGET')
selected_vars = iv_summary[iv_summary['info_value'] > 0.1]['variable'].tolist()
print(f"Selected features: {len(selected_vars)}")

In [ ]:
train_woe = sc.woebin_ply(app_train[selected_vars + ['TARGET']], bins)
selected_vars_woe = [var + '_woe' for var in selected_vars]
X_train = train_woe[selected_vars_woe]
y_train = train_woe['TARGET']

In [ ]:
app_train_B = pd.get_dummies(app_train, columns=char_cols_train, drop_first=True)
X_train_final = app_train_B.drop(columns=['TARGET'])
scaler = StandardScaler()
X_train_scaled = pd.DataFrame(scaler.fit_transform(X_train_final), columns=X_train_final.columns)

In [ ]:
lr_baseline = LogisticRegression(C=1, class_weight='balanced', random_state=42)
lr_baseline.fit(X_train, y_train)

rf_model = RandomForestClassifier(
    n_estimators=100, max_depth=10, min_samples_leaf=50,
    class_weight='balanced', random_state=42, n_jobs=-1
)
rf_model.fit(X_train_final, y_train)

feature_importance_df = pd.DataFrame({
    'Feature': X_train_final.columns,
    'Importance': rf_model.feature_importances_
}).sort_values(by='Importance', ascending=False)

feature_selected = feature_importance_df['Feature'].head(15).tolist()
X_train_input = X_train_scaled[feature_selected]

km_final = KMeans(n_clusters=5, init='k-means++', random_state=42, n_init=10)
km_final.fit(X_train_input)
print("All models trained!")

In [ ]:
joblib.dump(rf_model, 'rf_model.joblib')
joblib.dump(km_final, 'kmeans_model.joblib')
joblib.dump(scaler, 'scaler.joblib')
joblib.dump(lr_baseline, 'lr_model.joblib')

with open('woe_bins.pkl', 'wb') as f:
    pickle.dump(bins, f)
with open('selected_vars.pkl', 'wb') as f:
    pickle.dump(selected_vars, f)
with open('feature_selected.pkl', 'wb') as f:
    pickle.dump(feature_selected, f)

print("All models saved!")

In [ ]:
# LR predictions (using WOE features)
lr_probs = lr_baseline.predict_proba(X_train)[:, 1]

# RF predictions (using full features)
rf_probs = rf_model.predict_proba(X_train_final)[:, 1]

# KMeans clustering
clusters = km_final.predict(X_train_scaled[feature_selected])

# Save results
results = pd.DataFrame({
    'sk_id_curr': app_train['SK_ID_CURR'],
    'target': app_train['TARGET'],
    'lr_default_probability': lr_probs,
    'rf_default_probability': rf_probs,
    'cluster': clusters
})

results.to_csv('predictions.csv', index=False)
print(f"Saved {len(results)} predictions!")
print(results.head())